# Project 3 - manual notebook

In [1]:
import os, json, time, requests
from kafka.admin import KafkaAdminClient
import psycopg2

# ── Service endpoints ────────────────────────────────────────────────────────
AIRFLOW_URL  = "http://airflow:8080"
CONNECT_URL  = "http://connect:8083"
BOOTSTRAP    = "kafka:9092"
DAG_ID       = "project3_pipeline"
CONNECTOR_ID = "postgres-connector"
AUTH         = ("admin", "admin")
DAGS_DIR     = "/home/jovyan/project/dags"

# ── PostgreSQL helper ────────────────────────────────────────────────────────
PG = dict(
    host="postgres",
    port=5432,
    dbname="sourcedb",
    user="cdc_user",
    password="admin",
)

def pg(sql, params=None, fetch=False):
    conn = psycopg2.connect(**PG)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(sql, params)
    result = cur.fetchall() if fetch else None
    cur.close()
    conn.close()
    return result

# ── Airflow REST API helpers ─────────────────────────────────────────────────
def af_get(path):
    return requests.get(f"{AIRFLOW_URL}{path}", auth=AUTH)

def af_post(path, payload):
    return requests.post(f"{AIRFLOW_URL}{path}", json=payload, auth=AUTH)

def af_patch(path, payload):
    return requests.patch(f"{AIRFLOW_URL}{path}", json=payload, auth=AUTH)

def af_delete(path):
    return requests.delete(f"{AIRFLOW_URL}{path}", auth=AUTH)

def trigger_dag(dag_id=DAG_ID, logical_date=None):
    payload = {"dag_run_id": f"notebook__{int(time.time())}"}
    if logical_date:
        payload["logical_date"] = logical_date

    r = af_post(f"/api/v1/dags/{dag_id}/dagRuns", payload)

    if r.status_code in (200, 201):
        data = r.json()
        print(f"Triggered run_id={data['dag_run_id']} state={data['state']}")
        return data["dag_run_id"]

    print(f"Trigger error {r.status_code}: {r.text[:500]}")
    return None

def poll_run(run_id, dag_id=DAG_ID, timeout=600, interval=10):
    print(f"Polling {dag_id} / {run_id} ...")

    for elapsed in range(0, timeout, interval):
        r = af_get(f"/api/v1/dags/{dag_id}/dagRuns/{run_id}")

        if r.status_code != 200:
            print(f"Poll error {r.status_code}: {r.text[:300]}")
            return "error"

        state = r.json().get("state", "unknown")
        print(f"  [{elapsed:>3}s] {state}")

        if state in ("success", "failed"):
            return state

        time.sleep(interval)

    print("TIMEOUT")
    return "timeout"

def show_tasks(run_id, dag_id=DAG_ID):
    r = af_get(f"/api/v1/dags/{dag_id}/dagRuns/{run_id}/taskInstances")

    if r.status_code != 200:
        print(f"Task fetch error {r.status_code}: {r.text[:500]}")
        return

    tasks = r.json().get("task_instances", [])
    icons = {
        "success": "✓",
        "failed": "✗",
        "skipped": "⏭",
        "running": "⟳",
        "upstream_failed": "!",
        None: "?",
    }

    print(f"Task states for {run_id}:")
    for t in sorted(tasks, key=lambda x: x.get("start_date") or ""):
        s = t.get("state")
        print(f"  {icons.get(s, '?')} {t['task_id']:<28} {s}")

print("Helpers ready.")


Helpers ready.


## 1. Service health checks


In [2]:
# ── Verify services ──────────────────────────────────────────────────────────

r = requests.get(f"{AIRFLOW_URL}/api/v1/health", auth=AUTH)
print("Airflow health HTTP:", r.status_code)
print(r.text[:500])

if r.status_code == 401:
    raise RuntimeError(
        "Airflow API returned 401 Unauthorized. "
        "Add AIRFLOW__API__AUTH_BACKENDS=airflow.api.auth.backend.basic_auth,airflow.api.auth.backend.session "
        "to the Airflow service in compose.yml and recreate the Airflow container."
    )

h = r.json()
print(
    f"Airflow scheduler={h.get('scheduler', {}).get('status')} "
    f"db={h.get('metadatabase', {}).get('status')}"
)

r = requests.get(f"{CONNECT_URL}/")
print("Connect HTTP:", r.status_code)
print("Connect version:", r.json().get("version"))

ver = pg("SELECT version();", fetch=True)
print(f"Postgres {ver[0][0][:55]}...")

wal = pg("SHOW wal_level;", fetch=True)
assert wal[0][0] == "logical", "wal_level must be logical for CDC!"
print(f"wal_level {wal[0][0]} ✓")

print("\nAll services reachable.")


Airflow health HTTP: 200
{
  "dag_processor": {
    "latest_dag_processor_heartbeat": null,
    "status": null
  },
  "metadatabase": {
    "status": "healthy"
  },
  "scheduler": {
    "latest_scheduler_heartbeat": "2026-05-03T03:08:45.485897+00:00",
    "status": "healthy"
  },
  "triggerer": {
    "latest_triggerer_heartbeat": null,
    "status": null
  }
}

Airflow scheduler=healthy db=healthy
Connect HTTP: 200
Connect version: 3.9.0
Postgres PostgreSQL 16.13 on x86_64-pc-linux-musl, compiled by g...
wal_level logical ✓

All services reachable.


## 2. Check source tables


In [3]:
tables = pg("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
ORDER BY table_name;
""", fetch=True)

print("Public tables:", tables)

for table in ["customers", "drivers"]:
    try:
        n = pg(f"SELECT COUNT(*) FROM public.{table};", fetch=True)[0][0]
        print(f"{table}: {n} rows")
    except Exception as e:
        print(f"{table}: missing/unreadable -> {e}")

print("\nIf missing, run in terminal:")
print("cd /home/jovyan/project && python seed.py")


Public tables: [('customers',), ('drivers',)]
customers: 50 rows
drivers: 28 rows

If missing, run in terminal:
cd /home/jovyan/project && python seed.py


## 3. Register/check Debezium connector


In [4]:
CONNECTOR_ID = "postgres-connector"

connector_config = {
    "name": CONNECTOR_ID,
    "config": {
        "connector.class": "io.debezium.connector.postgresql.PostgresConnector",
        "plugin.name": "pgoutput",
        "database.hostname": "postgres",
        "database.port": "5432",
        "database.user": os.getenv("PG_USER", "cdc_user"),
        "database.password": os.getenv("PG_PASSWORD", "admin"),
        "database.dbname": os.getenv("PG_DB", "sourcedb"),
        "topic.prefix": "dbserver1",
        "table.include.list": "public.customers,public.drivers",
        "slot.name": "debezium_project3",
        "publication.name": "dbz_publication",
        "snapshot.mode": "initial",
        "tombstones.on.delete": "true",
    },
}

def get_connector_status():
    r = requests.get(f"{CONNECT_URL}/connectors/{CONNECTOR_ID}/status")
    print("Connector status HTTP:", r.status_code)

    if r.status_code == 200:
        data = r.json()
        print(json.dumps(data, indent=2))
        return data

    print(r.text)
    return None


def register_connector():
    existing = requests.get(f"{CONNECT_URL}/connectors/{CONNECTOR_ID}/status")

    if existing.status_code == 200:
        print(f"Connector {CONNECTOR_ID} already exists.")
        return get_connector_status()

    r = requests.post(
        f"{CONNECT_URL}/connectors",
        headers={"Content-Type": "application/json"},
        json=connector_config,
    )

    print("Register connector HTTP:", r.status_code)
    print(r.text[:1000])

    time.sleep(5)
    return get_connector_status()


status = register_connector()

Connector postgres-connector already exists.
Connector status HTTP: 200
{
  "name": "postgres-connector",
  "connector": {
    "state": "RUNNING",
    "worker_id": "172.19.0.7:8083"
  },
  "tasks": [
    {
      "id": 0,
      "state": "RUNNING",
      "worker_id": "172.19.0.7:8083"
    }
  ],
  "type": "source"
}


## 4. Confirm Kafka topics


In [5]:
admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
topics = sorted(admin.list_topics())
admin.close()

print("Kafka topics:")
for t in topics:
    print(" -", t)

expected = ["dbserver1.public.customers", "dbserver1.public.drivers", "taxi-trips"]
print("\nExpected topic check:")
for t in expected:
    print(f"{t}: {'OK' if t in topics else 'MISSING'}")

if "taxi-trips" not in topics:
    print("\nRun before taxi pipeline:")
    print("cd /home/jovyan/project && python produce.py --loop")


Kafka topics:
 - __consumer_offsets
 - _connect_configs
 - _connect_offsets
 - _connect_statuses
 - dbserver1.public.customers
 - dbserver1.public.drivers

Expected topic check:
dbserver1.public.customers: OK
dbserver1.public.drivers: OK
taxi-trips: MISSING

Run before taxi pipeline:
cd /home/jovyan/project && python produce.py --loop


## 5. Create Airflow HTTP connection for connector sensor


In [6]:
conn_id = "connect"

payload = {
    "connection_id": conn_id,
    "conn_type": "http",
    "host": "connect",
    "port": 8083,
    "schema": "http",
}

existing = af_get(f"/api/v1/connections/{conn_id}")

if existing.status_code == 200:
    print("Airflow connection already exists:", conn_id)
elif existing.status_code == 404:
    r = af_post("/api/v1/connections", payload)
    print("Create Airflow connection HTTP:", r.status_code)
    print(r.text[:500])
else:
    print("Unexpected HTTP:", existing.status_code)
    print(existing.text[:500])


Airflow connection already exists: connect


## 6. Check DAG visibility and unpause


In [7]:
r = requests.get(f"{AIRFLOW_URL}/api/v1/dags/{DAG_ID}", auth=AUTH)
print("DAG lookup HTTP:", r.status_code)

if r.status_code == 200:
    data = r.json()
    print(json.dumps(data, indent=2)[:1500])
    r2 = requests.patch(
        f"{AIRFLOW_URL}/api/v1/dags/{DAG_ID}",
        auth=AUTH,
        json={"is_paused": False},
    )
    print("\nUnpause HTTP:", r2.status_code)
    print(r2.text[:500])
else:
    print(r.text)
    print("Check: docker exec -it airflow airflow dags list-import-errors")


DAG lookup HTTP: 200
{
  "dag_display_name": "project3_pipeline",
  "dag_id": "project3_pipeline",
  "default_view": "grid",
  "description": "Project 3 CDC and taxi lakehouse pipeline",
  "file_token": "Ii9vcHQvYWlyZmxvdy9kYWdzL3Byb2plY3QzX3BpcGVsaW5lLnB5Ig.NeXIcQEacZp0fo6fLpoyJLwY0Vg",
  "fileloc": "/opt/airflow/dags/project3_pipeline.py",
  "has_import_errors": false,
  "has_task_concurrency_limits": false,
  "is_active": true,
  "is_paused": false,
  "is_subdag": false,
  "last_expired": null,
  "last_parsed_time": "2026-05-03T03:08:40.584410+00:00",
  "last_pickled": null,
  "max_active_runs": 1,
  "max_active_tasks": 16,
  "max_consecutive_failed_dag_runs": 0,
  "next_dagrun": "2026-05-03T03:00:00+00:00",
  "next_dagrun_create_after": null,
  "next_dagrun_data_interval_end": "2026-05-03T03:05:00+00:00",
  "next_dagrun_data_interval_start": "2026-05-03T03:00:00+00:00",
  "owners": [
    "airflow"
  ],
  "pickle_id": null,
  "root_dag_id": null,
  "schedule_interval": {
    "__type

## 7. Trigger DAG and poll result


In [8]:
def trigger_dag(dag_id=DAG_ID):
    run_id = f"notebook__{int(time.time())}"
    r = requests.post(
        f"{AIRFLOW_URL}/api/v1/dags/{dag_id}/dagRuns",
        auth=AUTH,
        json={"dag_run_id": run_id},
    )
    print("Trigger HTTP:", r.status_code)
    print(r.text[:1000])
    if r.status_code in (200, 201):
        return r.json()["dag_run_id"]
    return None

def poll_run(run_id, dag_id=DAG_ID, timeout=600, interval=10):
    print(f"Polling {dag_id} / {run_id}")
    for elapsed in range(0, timeout + interval, interval):
        r = requests.get(f"{AIRFLOW_URL}/api/v1/dags/{dag_id}/dagRuns/{run_id}", auth=AUTH)
        state = r.json().get("state")
        print(f"[{elapsed:>4}s] state={state}")
        if state in ("success", "failed"):
            return state
        time.sleep(interval)
    return "timeout"

def show_task_states(run_id, dag_id=DAG_ID):
    r = requests.get(
        f"{AIRFLOW_URL}/api/v1/dags/{dag_id}/dagRuns/{run_id}/taskInstances",
        auth=AUTH,
    )
    tasks = r.json().get("task_instances", [])
    print("Task states:")
    for t in sorted(tasks, key=lambda x: x["task_id"]):
        print(f"  {t['task_id']:<25} {t.get('state')}")

run_id = trigger_dag()
if run_id:
    final_state = poll_run(run_id)
    print("\nFinal state:", final_state)
    show_task_states(run_id)


Trigger HTTP: 200
{
  "conf": {},
  "dag_id": "project3_pipeline",
  "dag_run_id": "notebook__1777777728",
  "data_interval_end": "2026-05-03T03:05:00+00:00",
  "data_interval_start": "2026-05-03T03:00:00+00:00",
  "end_date": null,
  "execution_date": "2026-05-03T03:08:48.286137+00:00",
  "external_trigger": true,
  "last_scheduling_decision": null,
  "logical_date": "2026-05-03T03:08:48.286137+00:00",
  "note": null,
  "run_type": "manual",
  "start_date": null,
  "state": "queued"
}

Polling project3_pipeline / notebook__1777777728
[   0s] state=queued
[  10s] state=queued
[  20s] state=queued
[  30s] state=running
[  40s] state=running
[  50s] state=running
[  60s] state=running
[  70s] state=running
[  80s] state=running
[  90s] state=running
[ 100s] state=running
[ 110s] state=running
[ 120s] state=running
[ 130s] state=running
[ 140s] state=running
[ 150s] state=running
[ 160s] state=running
[ 170s] state=running
[ 180s] state=running
[ 190s] state=running
[ 200s] state=running


## 8. Start Spark for Iceberg evidence queries


In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("project3-demo")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0",
            "org.apache.iceberg:iceberg-aws-bundle:1.10.0",
        ])
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.demo", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.demo.type", "rest")
    .config("spark.sql.catalog.demo.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.demo.warehouse", "s3a://warehouse")
    .config("spark.sql.catalog.demo.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.demo.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.demo.s3.path-style-access", "true")
    .config("spark.sql.catalog.demo.s3.access-key-id", os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.demo.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.demo.client.region", "us-east-1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark ready:", spark.version)


Spark ready: 4.1.0


In [11]:
spark.sql("""
SELECT *
FROM demo.silver.customers
WHERE email = 'delete.test@example.com'
""").show(truncate=False)

+---+--------------------+-----------------------+-------+--------------------------+---------------+--------+----------------+
|id |name                |email                  |country|created_at                |cdc_event_ts_ms|cdc_lsn |cdc_kafka_offset|
+---+--------------------+-----------------------+-------+--------------------------+---------------+--------+----------------+
|78 |Delete Test Customer|delete.test@example.com|Estonia|2026-05-03 03:11:59.848853|1777777920401  |26964344|50              |
+---+--------------------+-----------------------+-------+--------------------------+---------------+--------+----------------+



In [15]:
spark.sql("""
SELECT *
FROM demo.silver.customers
WHERE email = 'delete.test@example.com'
""").show(truncate=False)

+---+----+-----+-------+----------+---------------+-------+----------------+
|id |name|email|country|created_at|cdc_event_ts_ms|cdc_lsn|cdc_kafka_offset|
+---+----+-----+-------+----------+---------------+-------+----------------+
+---+----+-----+-------+----------+---------------+-------+----------------+



In [14]:
spark.sql("""
SELECT op, before_json, after_json, is_tombstone
FROM demo.bronze.customers_cdc
WHERE pk_id = 78
ORDER BY kafka_offset
""").show(truncate=False)

+---+------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------+------------+
|op |before_json                                                 |after_json                                                                                                                   |is_tombstone|
+---+------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------+------------+
|c  |NULL                                                        |{"id":78,"name":"Delete Test Customer","email":"delete.test@example.com","country":"Estonia","created_at":"1777777919848853"}|false       |
|d  |{"id":78,"name":"","email":"","country":"","created_at":"0"}|NULL                                                                                                          

## 9. Validate CDC Silver against PostgreSQL


In [16]:
pg_customers = pg("SELECT COUNT(*) FROM public.customers;", fetch=True)[0][0]
pg_drivers = pg("SELECT COUNT(*) FROM public.drivers;", fetch=True)[0][0]

silver_customers = spark.sql("SELECT COUNT(*) AS n FROM demo.silver.customers").collect()[0]["n"]
silver_drivers = spark.sql("SELECT COUNT(*) AS n FROM demo.silver.drivers").collect()[0]["n"]

print("PostgreSQL customers:", pg_customers)
print("Silver customers:    ", silver_customers)
print("PostgreSQL drivers:  ", pg_drivers)
print("Silver drivers:      ", silver_drivers)

assert pg_customers == silver_customers, "customers count mismatch"
assert pg_drivers == silver_drivers, "drivers count mismatch"
print("\n✓ Silver CDC matches PostgreSQL row counts")


PostgreSQL customers: 50
Silver customers:     50
PostgreSQL drivers:   28
Silver drivers:       28

✓ Silver CDC matches PostgreSQL row counts


## 10. Spot-check sample rows


In [17]:
print("PostgreSQL customers sample:")
for row in pg("SELECT id, name, email, country FROM public.customers ORDER BY id LIMIT 3;", fetch=True):
    print(row)

print("\nSilver customers sample:")
spark.sql("SELECT id, name, email, country FROM demo.silver.customers ORDER BY id LIMIT 3").show(truncate=False)

print("\nPostgreSQL drivers sample:")
for row in pg("SELECT id, name, license_number, rating, city, active FROM public.drivers ORDER BY id LIMIT 3;", fetch=True):
    print(row)

print("\nSilver drivers sample:")
spark.sql("SELECT id, name, license_number, rating, city, active FROM demo.silver.drivers ORDER BY id LIMIT 3").show(truncate=False)


PostgreSQL customers sample:
(1, 'Alice Mets', 'updated_1_641@example.com', 'Estonia')
(2, 'Bob Virtanen', 'updated_2_332@test.net', 'Netherlands')
(3, 'Carol Ozols', 'updated_3_672@example.com', 'Brazil')

Silver customers sample:
+---+------------+-------------------------+-----------+
|id |name        |email                    |country    |
+---+------------+-------------------------+-----------+
|1  |Alice Mets  |updated_1_641@example.com|Estonia    |
|2  |Bob Virtanen|updated_2_332@test.net   |Netherlands|
|3  |Carol Ozols |updated_3_672@example.com|Brazil     |
+---+------------+-------------------------+-----------+


PostgreSQL drivers sample:
(1, 'Tom Driver', 'TLC-10001', Decimal('4.69'), 'Manhattan', True)
(3, 'Mike Road', 'TLC-10003', Decimal('3.70'), 'Bronx', False)
(5, 'Jake Taxi', 'TLC-10005', Decimal('4.78'), 'Manhattan', True)

Silver drivers sample:
+---+----------+--------------+------+---------+------+
|id |name      |license_number|rating|city     |active|
+---+---

## 11. Table schemas


In [19]:
tables = [
    "demo.bronze.customers_cdc",
    "demo.bronze.drivers_cdc",
    "demo.silver.customers",
    "demo.silver.drivers",
    "demo.bronze.stg_taxi",
    "demo.silver.fct_taxi_trip",
    "demo.gold.analytical_taxi_trips",
    "demo.gold.taxi_zone_hourly_metrics",
]

for table in tables:
    print("\n" + "=" * 80)
    print(table)
    try:
        spark.sql(f"DESCRIBE TABLE {table}").show(100, truncate=False)
    except Exception as e:
        print("Not available yet:", e)



demo.bronze.customers_cdc
+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|pk_id          |bigint   |NULL   |
|topic          |string   |NULL   |
|kafka_partition|int      |NULL   |
|kafka_offset   |bigint   |NULL   |
|kafka_timestamp|timestamp|NULL   |
|is_tombstone   |boolean  |NULL   |
|op             |string   |NULL   |
|lsn            |bigint   |NULL   |
|event_ts_ms    |bigint   |NULL   |
|key_json       |string   |NULL   |
|before_json    |string   |NULL   |
|after_json     |string   |NULL   |
|value_json     |string   |NULL   |
+---------------+---------+-------+


demo.bronze.drivers_cdc
+---------------+---------+-------+
|col_name       |data_type|comment|
+---------------+---------+-------+
|pk_id          |bigint   |NULL   |
|topic          |string   |NULL   |
|kafka_partition|int      |NULL   |
|kafka_offset   |bigint   |NULL   |
|kafka_timestamp|timestamp|NULL   |
|is_tombstone   |boolean  |NULL   |
|op        

## 12. Iceberg snapshot history


In [20]:
for table in ["demo.silver.customers", "demo.silver.drivers"]:
    print("\n" + "=" * 80)
    print(table + ".snapshots")
    try:
        spark.sql(f"SELECT * FROM {table}.snapshots ORDER BY committed_at DESC").show(20, truncate=False)
    except Exception as e:
        print("Snapshot query failed:", e)



demo.silver.customers.snapshots
+-----------------------+-------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                |summary                                                    

## 13. Time travel example


In [ ]:
SNAPSHOT_ID = 4200449033602875954

if SNAPSHOT_ID is None:
    print("Set SNAPSHOT_ID from snapshot history, then rerun this cell.")
else:
    spark.sql(f"""
        SELECT COUNT(*) AS customer_count_at_snapshot
        FROM demo.silver.customers VERSION AS OF {SNAPSHOT_ID}
    """).show()


In [ ]:
spark.sql("""
SELECT COUNT(*) AS customers_old_count
FROM demo.silver.customers VERSION AS OF 4200449033602875954
""").show()

spark.sql("""
SELECT COUNT(*) AS customers_current_count
FROM demo.silver.customers
""").show()

## 14. Idempotency evidence


In [18]:
before = {
    "customers": spark.sql("SELECT COUNT(*) AS n FROM demo.silver.customers").collect()[0]["n"],
    "drivers": spark.sql("SELECT COUNT(*) AS n FROM demo.silver.drivers").collect()[0]["n"],
}
print("Before:", before)

run_id = trigger_dag()
if run_id:
    final_state = poll_run(run_id)
    show_task_states(run_id)

after = {
    "customers": spark.sql("SELECT COUNT(*) AS n FROM demo.silver.customers").collect()[0]["n"],
    "drivers": spark.sql("SELECT COUNT(*) AS n FROM demo.silver.drivers").collect()[0]["n"],
}
print("After:", after)

assert before == after, "Idempotency check failed: row counts changed"
print("\n✓ Idempotency check passed")


Before: {'customers': 50, 'drivers': 28}
Trigger HTTP: 200
{
  "conf": {},
  "dag_id": "project3_pipeline",
  "dag_run_id": "notebook__1777779021",
  "data_interval_end": "2026-05-03T03:30:00+00:00",
  "data_interval_start": "2026-05-03T03:25:00+00:00",
  "end_date": null,
  "execution_date": "2026-05-03T03:30:21.271698+00:00",
  "external_trigger": true,
  "last_scheduling_decision": null,
  "logical_date": "2026-05-03T03:30:21.271698+00:00",
  "note": null,
  "run_type": "manual",
  "start_date": null,
  "state": "queued"
}

Polling project3_pipeline / notebook__1777779021
[   0s] state=queued
[  10s] state=queued
[  20s] state=queued
[  30s] state=queued
[  40s] state=queued
[  50s] state=queued
[  60s] state=queued
[  70s] state=queued
[  80s] state=queued
[  90s] state=queued
[ 100s] state=queued
[ 110s] state=queued
[ 120s] state=queued
[ 130s] state=queued
[ 140s] state=queued
[ 150s] state=queued
[ 160s] state=queued
[ 170s] state=running
[ 180s] state=running
[ 190s] state=run

## 15. Taxi pipeline evidence


In [21]:
for table in [
    "demo.bronze.stg_taxi",
    "demo.silver.fct_taxi_trip",
    "demo.gold.analytical_taxi_trips",
    "demo.gold.taxi_zone_hourly_metrics",
]:
    print("\n" + "=" * 80)
    print(table)
    spark.sql(f"SELECT COUNT(*) AS n FROM {table}").show()
    spark.sql(f"SELECT * FROM {table} LIMIT 5").show(truncate=False)



demo.bronze.stg_taxi
+---+
|  n|
+---+
|177|
+---+

+-----------------------+---+------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|kafka_time             |key|offset|partition|value                                                                                                                                                                                                                                                                                                                                                                             